In [105]:
def create_bqm(Q, offset):
    BQM = dimod.BinaryQuadraticModel.empty(dimod.BINARY)
    for (v1,v2), value in Q.items():
        if v1 == v2:
            BQM.add_linear(v1,value)
        else:
            BQM.add_quadratic(v1,v2,value)
    BQM.offset += offset
    return BQM

In [106]:
def advantage(input_file, output_file, qubo_col, embedding_col, num_reads, annealing_time):
    solver = DWaveSampler()
    df = pd.read_csv(input_file)
    n = len(df.index)
    if Path(output_file).is_file():
        new_df = pd.read_csv(output_file)
        n = len(new_df.index) 
    else:
        new_df = pd.DataFrame()
        if n==30:
            new_df['graph_num'] = df['graph_num']
    sample = []
    qpu_access_time = []
    optimal = []
    
    count = 0
    
    for i in range(n):
        opt = 'N'
        qubo_str = df[qubo_col][i]
        qubo = ast.literal_eval(qubo_str)
        embedding_str = df[embedding_col][i]
        embedding = ast.literal_eval(embedding_str)
        offset = df['offset'][i]
        
        BQM = create_bqm(qubo, offset)
        
        sampler = FixedEmbeddingComposite(solver, embedding)
        sampleset = sampler.sample(BQM,num_reads=num_reads,annealing_time=annealing_time)
        #print(sampleset.first)
        qpu_time = sampleset.info['timing']['qpu_access_time']
        s = sampleset.first[0]
        qpu_access_time.append(qpu_time)
        sample.append(s)
        energy = sampleset.first[1]
        if energy == 0:
            opt = 'Y'
            count+=1
        optimal.append(opt)
    
    new_df[f'sample_{num_reads}_{annealing_time}'] = sample
    new_df[f'qpu_access_time_{num_reads}_{annealing_time}'] = qpu_access_time
    new_df[f'optimal_{num_reads}_{annealing_time}'] = optimal
    #new_df.drop(df.columns[df.columns.str.contains('unnamed',case = False)],axis = 1, inplace = True)
    print(count/n)
    new_df.to_csv(output_file)
   

In [ ]:
import datetime as dt
import pandas as pd
import networkx as nx
import json
import ast
import time
import dimod
from pathlib import Path
from dwave.system import DWaveSampler, EmbeddingComposite
from dwave.system import FixedEmbeddingComposite
from dwave.embedding import chain_breaks
#advantage('qubos/unconstrained/apg5_qubo.csv', 'Advantage1_Pegasus/apg5_pegasus.csv','qubo','pegasus_embedding',10000,100)
advantage('qubos/unconstrained/apg4_qubo.csv', 'Advantage2_Prototype_Zephyr/apg4_zephyr.csv','qubo','zephyr_embedding',10000,100)


In [ ]:
sampleset.info

In [4]:
DWaveSampler().properties['chip_id']

'Advantage_system6.3'

In [ ]:
DWaveSampler().properties